In [2]:
%pip install -q openpyxl



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd 
import numpy as np 

path = "h252.xlsx" ; sheet = "H252"

df = pd.read_excel(path, sheet_name=sheet)

df.columns = [str(c).strip() for c in df.columns]

df.head()


,DUID,PID,DUPERSID,PANEL,YEARIND,SAQRDS24,ALL5RDS,DIED,INST,MILITARY,...,RXOSRY1,RXOSRY2,RXPTRY1,RXPTRY2,RXOTHY1,RXOTHY2,VARPSU,VARSTR,LONGWT,LSAQWT
0,2790002,101,2790002101,27,1,1,1,0,0,0,...,0,0,477,82,0,0,1,2019,28764.957562,38800.049850
1,2790002,102,2790002102,27,1,0,1,0,0,0,...,0,0,0,0,0,0,1,2019,73143.190424,0.000000
2,2790004,101,2790004101,27,1,0,1,0,0,0,...,0,0,0,0,0,0,1,2084,52401.895364,71295.789233
3,2790006,101,2790006101,27,1,1,1,0,0,0,...,0,0,0,0,0,0,1,2113,26781.859335,27214.872872
4,2790006,102,2790006102,27,1,0,1,0,0,0,...,0,0,0,0,0,0,1,2113,42444.547450,0.000000


In [ ]:


special_na = {-1: np.nan, -7: np.nan, -8: np.nan, -9: np.nan, -15: np.nan}

df = df.replace(special_na)

df = df[(df.get("YEARIND",1)==1) & (df.get("ALL5RDS",1)==1)].copy()

id_cols     = [c for c in ["DUPERSID","PANEL"] if c in df.columns]

weight_cols = [c for c in ["LONGWT","PERWT22F","PERWT23F"] if c in df.columns]

design_cols = [c for c in ["VARSTR","VARPSU"] if c in df.columns]

#识别样本标记列（面板期间的进入/退出等标志）。
sample_cols = [c for c in ["YEARIND","ALL5RDS","DIED","INST","MILITARY","ENTRSRVY","LEFTUS","OTHER"] if c in df.columns]

#列出第二年目标（总支出、住院次数、急诊次数、死亡等）中，实际在表里的列名。
y2_targets = [c for c in ["TOTEXPY2","IPDISY2","ERTOTY2","DIED"] if c in df.columns]

#用后缀识别 Y1 列（2022 年的年度/轮次统一后的字段）。
is_Y1 = df.columns.to_series().str.upper().str.endswith("Y1")

#用后缀识别 Y2 列（2023 年的年度/轮次统一后的字段）。
is_Y2 = df.columns.to_series().str.upper().str.endswith("Y2")

#列出不随年份变化的常量/背景列（可作为特征使用）。
const_ok = [c for c in ["SEX","RACEV1X","RACEX","HISPANX","EDUC"] if c in df.columns]

#初步收集 Y1 特征候选：所有以 Y1 结尾的列 + 常量列 + Y1 年末保险相关列。
y1_feature_candidates = sorted(set(df.columns[is_Y1].tolist() + const_ok + [c for c in df.columns if c.startswith("INSCY1")]))

#整理基础列：ID、权重、设计、样本标记，后续两个表都带着它们。
base_cols = id_cols + weight_cols + design_cols + sample_cols

#定义train 表的列：基础列 + Y1 特征候选 +（附带）Y2 目标（提醒：建模时别把 Y2 当特征）。
train_cols = base_cols + y1_feature_candidates + y2_targets

#定义test/评估 表的列：基础列 + Y2 目标。
test_cols  = base_cols + y2_targets

#生成 train 表（只保留确实存在的列）。
train_df = df.loc[:, [c for c in train_cols if c in df.columns]].copy()

#生成 test/评估 表（只保留确实存在的列）。
test_df  = df.loc[:, [c for c in test_cols  if c in df.columns]].copy()

# 可选：保存两张表（供后续建模/统计使用）
train_df.to_parquet("hc252_train_y1_features.parquet", index=False)

test_df.to_parquet("hc252_test_y2_targets.parquet", index=False)